# Olist 巴西电商销售数据分析项目

**项目目标：** 基于 Olist 公开电商数据，完成数据读取、质量检查、多表关联、销售分析、商品品类分析、地区销售分析、客户行为分析及业务可视化。

**数据口径说明**
- 核心销售分析以 `order_status == "delivered"` 的已送达订单为范围。
- 销售额使用商品明细表中的 `price`，不包含运费 `freight_value`。
- 商品明细表一行代表一条商品明细记录；数据集中没有原始商品购买数量字段，因此不将人为构造的 `quantity=1` 解释为实际购买件数。
- 客户分析使用 `customer_unique_id` 识别客户；订单量使用 `order_id` 去重。
- 所有业务结论以本 Notebook 实际运行结果为准，不预填未经验证的数字。

**运行说明：** 默认 Notebook 位于 `notebooks/` 文件夹，CSV 文件位于 `data/raw/`，图表保存至 `images/`。

## 目录

1. 数据读取
2. 数据质量检查
3. 数据表合并
4. 数据清洗与分析数据集构建
5. 销售核心指标分析
6. 商品品类销售分析
7. 地区销售分析
8. 客户行为分析
9. 数据可视化
10. 项目总结与业务建议

## 1. 数据读取

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

# Notebook 位于 notebooks/，数据位于项目根目录下的 data/raw/
data_path = "../data/raw/"
image_path = "../images/"
os.makedirs(image_path, exist_ok=True)

# 读取分析所需数据表
orders = pd.read_csv(data_path + "olist_orders_dataset.csv")
items = pd.read_csv(data_path + "olist_order_items_dataset.csv")
products = pd.read_csv(data_path + "olist_products_dataset.csv")
customers = pd.read_csv(data_path + "olist_customers_dataset.csv")
category = pd.read_csv(data_path + "product_category_name_translation.csv")

tables = {
    "orders": orders,
    "items": items,
    "products": products,
    "customers": customers,
    "category_translation": category
}

for name, table in tables.items():
    print(f"{name}: {table.shape}")

display(orders.head())
display(items.head())

## 2. 数据质量检查

先检查数据结构、缺失值、重复记录和订单状态。缺失值不进行无差别填充或删除；是否处理取决于字段用途及业务含义。

In [ ]:
# 查看字段类型与非空数量
orders.info()
items.info()

# 缺失值数量及比例
for name, table in [("orders", orders), ("items", items),
                    ("products", products), ("customers", customers)]:
    missing = pd.DataFrame({
        "缺失数量": table.isna().sum(),
        "缺失比例(%)": (table.isna().mean() * 100).round(2)
    })
    print(f"\n{name} 缺失值检查")
    display(missing[missing["缺失数量"] > 0].sort_values("缺失数量", ascending=False))

# 重复记录与主键检查
print("orders 完全重复行数：", orders.duplicated().sum())
print("items 完全重复行数：", items.duplicated().sum())
print("orders 重复 order_id 数：", orders["order_id"].duplicated().sum())
print("items 重复 (order_id, order_item_id) 数：",
      items.duplicated(["order_id", "order_item_id"]).sum())

# 订单状态分布
display(orders["order_status"].value_counts(dropna=False).to_frame("订单数量"))

## 3. 数据表合并

以商品明细表 `items` 为主表，依次关联订单、商品、客户及品类英文翻译。使用 `validate='many_to_one'` 检查关联关系，避免维度表重复键导致行数膨胀。

In [ ]:
# 检查维度表关联键是否唯一
for name, table, key in [
    ("orders", orders, "order_id"),
    ("products", products, "product_id"),
    ("customers", customers, "customer_id"),
    ("category_translation", category, "product_category_name")
]:
    duplicated_keys = table[key].duplicated().sum()
    print(f"{name} 的 {key} 重复数：{duplicated_keys}")

# 多表关联
df = items.merge(
    orders, on="order_id", how="left", validate="many_to_one"
)
print("关联订单后：", df.shape)

df = df.merge(
    products, on="product_id", how="left", validate="many_to_one"
)
print("关联商品后：", df.shape)

df = df.merge(
    customers, on="customer_id", how="left", validate="many_to_one"
)
print("关联客户后：", df.shape)

df = df.merge(
    category, on="product_category_name", how="left", validate="many_to_one"
)
print("关联品类翻译后：", df.shape)

print("最终字段：")
print(df.columns.tolist())
display(df.head())

## 4. 数据清洗与分析数据集构建

构建统一的销售分析数据集。筛选已送达订单，并转换日期、金额字段；仅删除本次分析必需字段缺失的记录。

In [ ]:
sales_df = df.copy()

# 日期及金额字段转换
sales_df["order_purchase_timestamp"] = pd.to_datetime(
    sales_df["order_purchase_timestamp"], errors="coerce"
)
sales_df["price"] = pd.to_numeric(sales_df["price"], errors="coerce")
sales_df["freight_value"] = pd.to_numeric(
    sales_df["freight_value"], errors="coerce"
)

# 仅保留已送达订单
sales_df = sales_df.loc[
    sales_df["order_status"].eq("delivered")
].copy()

# 删除核心分析字段缺失的记录
required_columns = [
    "order_id", "customer_unique_id",
    "order_purchase_timestamp", "price"
]
before_rows = len(sales_df)
sales_df = sales_df.dropna(subset=required_columns).copy()
print("因核心字段缺失删除的商品明细行数：", before_rows - len(sales_df))

# 销售金额定义：商品价格，不含运费
sales_df["amount"] = sales_df["price"]
sales_df["year"] = sales_df["order_purchase_timestamp"].dt.year
sales_df["month"] = sales_df["order_purchase_timestamp"].dt.to_period("M")

# 地区与品类字段缺失不在此处删除，分别在对应分析模块处理
print("清洗后商品明细行数：", len(sales_df))
print("已送达订单数：", sales_df["order_id"].nunique())
print("客户数：", sales_df["customer_unique_id"].nunique())
print("地区字段缺失行数：", sales_df["customer_state"].isna().sum())
print("品类英文名缺失行数：", sales_df["product_category_name_english"].isna().sum())
display(sales_df.head())

## 5. 销售核心指标分析

指标口径：商品销售额 = `price` 求和；订单量 = 去重订单数；客户数 = 去重客户数；平均客单价 = 商品销售额 / 去重订单数。

In [ ]:
total_sales = sales_df["amount"].sum()
total_orders = sales_df["order_id"].nunique()
total_customers = sales_df["customer_unique_id"].nunique()
average_order_value = total_sales / total_orders if total_orders else np.nan

print("========== 销售核心指标 ==========")
print(f"商品销售额：{total_sales:,.2f} BRL")
print(f"已送达订单量：{total_orders:,}")
print(f"已送达客户数：{total_customers:,}")
print(f"平均订单商品金额（客单价）：{average_order_value:,.2f} BRL")

In [ ]:
# 按月汇总销售额、订单量和客户数
monthly_sales = sales_df.groupby("month").agg(
    销售额=("amount", "sum"),
    订单量=("order_id", "nunique"),
    客户数量=("customer_unique_id", "nunique")
).sort_index()

# 补齐连续月份，避免同比错位
all_months = pd.period_range(
    monthly_sales.index.min(), monthly_sales.index.max(), freq="M"
)
monthly_sales = monthly_sales.reindex(all_months)
monthly_sales.index.name = "月份"

# 没有订单的月份销售额、订单量记为0；客单价保持缺失
monthly_sales[["销售额", "订单量", "客户数量"]] = (
    monthly_sales[["销售额", "订单量", "客户数量"]].fillna(0)
)
monthly_sales["客单价"] = (
    monthly_sales["销售额"] /
    monthly_sales["订单量"].replace(0, np.nan)
)

monthly_sales["销售额环比增长率(%)"] = (
    monthly_sales["销售额"].pct_change(fill_method=None) * 100
)
monthly_sales["订单量环比增长率(%)"] = (
    monthly_sales["订单量"].pct_change(fill_method=None) * 100
)
monthly_sales["销售额同比增长率(%)"] = (
    monthly_sales["销售额"].pct_change(periods=12, fill_method=None) * 100
)

display(monthly_sales)

## 6. 商品品类销售分析

按商品英文品类统计销售额、订单量、商品明细数和客户数。缺失的英文品类统一标记为 `unknown`，避免静默丢失记录。

In [ ]:
category_df = sales_df.copy()
category_df["product_category_name_english"] = (
    category_df["product_category_name_english"].fillna("unknown")
)

category_sales = category_df.groupby(
    "product_category_name_english"
).agg(
    销售额=("amount", "sum"),
    订单量=("order_id", "nunique"),
    商品明细数量=("product_id", "count"),
    客户数量=("customer_unique_id", "nunique")
).sort_values("销售额", ascending=False)

category_sales["销售额占比(%)"] = (
    category_sales["销售额"] / category_sales["销售额"].sum() * 100
)

top10_category = category_sales.head(10)
best_category = category_sales["销售额"].idxmax()
best_category_sales = category_sales["销售额"].max()
top10_category_share = category_sales.head(10)["销售额"].sum() / category_sales["销售额"].sum() * 100

print("品类数量：", len(category_sales))
print("销售额最高品类：", best_category)
print(f"最高品类销售额：{best_category_sales:,.2f} BRL")
print(f"Top 10 品类销售额占比：{top10_category_share:.2f}%")
display(top10_category)

In [ ]:
# Top 10 品类销售额柱状图
plt.figure(figsize=(12, 7))
plt.barh(top10_category.index[::-1], top10_category["销售额"].values[::-1])
plt.title("Top 10 Product Categories by Sales")
plt.xlabel("Sales (BRL)")
plt.ylabel("Product Category")
plt.tight_layout()
plt.savefig(image_path + "category_sales_top10.png", dpi=300, bbox_inches="tight")
plt.show()

# Top 8 品类 + 其他品类占比
top8_sales = category_sales["销售额"].head(8)
other_sales = category_sales["销售额"].iloc[8:].sum()
pie_data = pd.concat([top8_sales, pd.Series({"其他品类": other_sales})])

plt.figure(figsize=(9, 9))
plt.pie(pie_data, labels=pie_data.index, autopct="%1.1f%%", startangle=90)
plt.title("Product Category Sales Share")
plt.tight_layout()
plt.savefig(image_path + "category_sales_share.png", dpi=300, bbox_inches="tight")
plt.show()

### 6.1 商品品类分析结论

根据上方实际运行结果填写和核对以下结论：

1. **核心品类表现：** 说明销售额排名第一的品类及其实际销售额，并结合 Top 10 排名指出其他主要品类。
2. **品类集中度：** 引用 Top 10 品类销售额占比，判断销售额是否集中于少数品类；同时说明长尾品类仍有一定贡献。
3. **运营建议：** 对核心品类关注供应稳定性和营销投入；对长尾品类结合订单量、客单价及利润（本项目尚未计算利润）进一步评估。
4. **口径限制：** 本模块分析的是商品销售额，不含运费，也未计算商品成本和利润。

## 7. 地区销售分析

以客户所在州 `customer_state` 为地区维度。地区缺失记录仅从本模块剔除，不影响其他模块的销售分析。

In [ ]:
region_df = sales_df.dropna(subset=["customer_state"]).copy()

region_sales = region_df.groupby("customer_state").agg(
    销售额=("amount", "sum"),
    订单量=("order_id", "nunique"),
    客户数量=("customer_unique_id", "nunique"),
    商品明细数量=("product_id", "count")
)

region_sales["客单价"] = region_sales["销售额"] / region_sales["订单量"]
region_sales["销售额占比(%)"] = (
    region_sales["销售额"] / region_sales["销售额"].sum() * 100
)
region_sales = region_sales.sort_values("销售额", ascending=False)

top10_region = region_sales.head(10)
best_region = region_sales["销售额"].idxmax()
best_region_sales = region_sales.loc[best_region, "销售额"]
top10_region_share = top10_region["销售额"].sum() / region_sales["销售额"].sum() * 100

print("有销售记录的州数量：", len(region_sales))
print("销售额最高州：", best_region)
print(f"该州销售额：{best_region_sales:,.2f} BRL")
print(f"Top 10 州销售额占比：{top10_region_share:.2f}%")
display(top10_region)

In [ ]:
# Top 10 州销售额
plt.figure(figsize=(12, 7))
plt.barh(top10_region.index[::-1], top10_region["销售额"].values[::-1])
plt.title("Top 10 States by Sales")
plt.xlabel("Sales (BRL)")
plt.ylabel("Customer State")
plt.tight_layout()
plt.savefig(image_path + "region_sales_top10.png", dpi=300, bbox_inches="tight")
plt.show()

# Top 10 州订单量
top10_orders = region_sales.sort_values("订单量", ascending=False).head(10)
plt.figure(figsize=(12, 7))
plt.barh(top10_orders.index[::-1], top10_orders["订单量"].values[::-1])
plt.title("Top 10 States by Order Volume")
plt.xlabel("Number of Orders")
plt.ylabel("Customer State")
plt.tight_layout()
plt.savefig(image_path + "region_orders_top10.png", dpi=300, bbox_inches="tight")
plt.show()

# 客单价排名设置最低订单量门槛，减少小样本干扰
MIN_ORDERS_FOR_AOV = 100
region_aov_filtered = region_sales.loc[
    region_sales["订单量"] >= MIN_ORDERS_FOR_AOV
].copy()
top10_aov = region_aov_filtered.sort_values("客单价", ascending=False).head(10)

print(f"客单价排名仅纳入订单量不少于 {MIN_ORDERS_FOR_AOV} 的州")
display(top10_aov[["客单价", "销售额", "订单量", "客户数量"]])

plt.figure(figsize=(12, 7))
plt.barh(top10_aov.index[::-1], top10_aov["客单价"].values[::-1])
plt.title("Top States by Average Order Value (Min. 100 Orders)")
plt.xlabel("Average Order Value (BRL)")
plt.ylabel("Customer State")
plt.tight_layout()
plt.savefig(image_path + "region_aov_top10.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# 地区汇总核对：地区总额应等于地区非空记录总额
print(f"地区汇总销售额：{region_sales['销售额'].sum():,.2f} BRL")
print(f"地区非空记录销售额：{region_df['amount'].sum():,.2f} BRL")
print("地区汇总订单量：", region_sales["订单量"].sum())
print("地区非空订单量：", region_df["order_id"].nunique())
print("地区字段缺失商品明细行数：", sales_df["customer_state"].isna().sum())

### 7.1 地区销售分析结论

请根据上方输出填写：

1. **核心市场：** 指出销售额最高的州及销售额。
2. **地区集中度：** 引用 Top 10 州销售额占比，判断销售是否集中于主要州。
3. **订单量与客单价：** 对比销售额、订单量和客单价排名，说明市场规模与单笔订单金额的差异。
4. **业务建议：** 对核心州维持服务和履约质量；对其他州结合订单量、客单价及物流成本制定差异化策略。

注意：本项目使用客户所在州，不代表卖家所在地；地区客单价排名设置了最低订单量门槛。

## 8. 客户行为分析

客户层级以 `customer_unique_id` 为准。先将商品明细汇总为订单级数据，避免一笔订单包含多个商品时被误算为多次购买。

In [ ]:
# 每行聚合为一笔订单
order_level = sales_df.groupby(
    ["order_id", "customer_unique_id"], as_index=False
).agg(
    order_date=("order_purchase_timestamp", "min"),
    order_amount=("amount", "sum")
)

order_level["purchase_month"] = order_level["order_date"].dt.to_period("M")
order_level = order_level.sort_values(
    ["customer_unique_id", "order_date", "order_id"]
)

print("订单级记录数：", len(order_level))
print("去重订单数：", order_level["order_id"].nunique())
print("客户数：", order_level["customer_unique_id"].nunique())
display(order_level.head())

In [ ]:
# 首次购买月份：在本数据集已送达订单范围内识别
customer_first_purchase = order_level.groupby(
    "customer_unique_id", as_index=False
).agg(
    first_purchase_date=("order_date", "min")
)
customer_first_purchase["first_purchase_month"] = (
    customer_first_purchase["first_purchase_date"].dt.to_period("M")
)

order_level = order_level.merge(
    customer_first_purchase[["customer_unique_id", "first_purchase_month"]],
    on="customer_unique_id",
    how="left",
    validate="many_to_one"
)

# 按月统计当月活跃客户、新客户及老客户
monthly_customer = order_level.groupby("purchase_month").agg(
    月活跃客户数=("customer_unique_id", "nunique"),
    新客户数=("customer_unique_id", lambda s: s[
        order_level.loc[s.index, "purchase_month"].eq(
            order_level.loc[s.index, "first_purchase_month"]
        )
    ].nunique())
)
monthly_customer["老客户数"] = (
    monthly_customer["月活跃客户数"] - monthly_customer["新客户数"]
)
display(monthly_customer)

In [ ]:
# 客户购买频次与累计消费金额
customer_summary = order_level.groupby("customer_unique_id").agg(
    购买订单数=("order_id", "nunique"),
    累计消费金额=("order_amount", "sum"),
    首次购买时间=("order_date", "min"),
    最近购买时间=("order_date", "max")
)

customer_summary["是否复购"] = customer_summary["购买订单数"] >= 2

total_customers = len(customer_summary)
repeat_customers = int(customer_summary["是否复购"].sum())
one_time_customers = int((customer_summary["购买订单数"] == 1).sum())
repurchase_rate = repeat_customers / total_customers * 100 if total_customers else np.nan
one_time_rate = one_time_customers / total_customers * 100 if total_customers else np.nan
avg_purchase_frequency = customer_summary["购买订单数"].mean()

print("客户总数：", total_customers)
print("复购客户数：", repeat_customers)
print(f"客户复购率：{repurchase_rate:.2f}%")
print(f"一次性购买客户占比：{one_time_rate:.2f}%")
print(f"客户平均购买订单数：{avg_purchase_frequency:.2f}")

In [ ]:
# 购买订单数分布
purchase_frequency = customer_summary["购买订单数"].value_counts().sort_index()
frequency_result = pd.DataFrame({
    "客户数量": purchase_frequency,
    "客户占比(%)": purchase_frequency / total_customers * 100
})
display(frequency_result)

In [ ]:
# 按累计消费金额的第90百分位数识别高价值客户
high_value_threshold = customer_summary["累计消费金额"].quantile(0.90)
customer_summary["客户类型"] = np.where(
    customer_summary["累计消费金额"] >= high_value_threshold,
    "高价值客户",
    "普通客户"
)

customer_segment = customer_summary.groupby("客户类型").agg(
    客户数量=("购买订单数", "count"),
    平均消费金额=("累计消费金额", "mean"),
    总消费金额=("累计消费金额", "sum"),
    平均购买订单数=("购买订单数", "mean")
)

customer_segment["客户数量占比(%)"] = (
    customer_segment["客户数量"] / total_customers * 100
)
customer_segment["消费金额贡献占比(%)"] = (
    customer_segment["总消费金额"] / customer_summary["累计消费金额"].sum() * 100
)

print(f"高价值客户识别门槛（P90）：{high_value_threshold:,.2f} BRL")
display(customer_segment)

### 8.1 客户行为分析结论

1. **新老客户分析：** 新客户定义为在本数据集已送达订单范围内首次出现的客户；老客户为此前月份已购买、当月再次购买的客户。该口径受数据集时间范围限制。
2. **客户复购率：** 复购客户定义为至少有两笔不同已送达订单的客户。复购率 = 复购客户数 / 客户总数。
3. **购买频次：** 使用订单级数据统计客户购买订单数，避免同一订单多条商品明细造成重复计数。
4. **高价值客户：** 使用累计商品消费金额的第 90 百分位数作为识别门槛。该分层是消费金额分层，不代表利润贡献；由于分位数并列，客户比例不一定恰好为 10%。

请依据上方实际输出填写具体数值，不要将历史运行结果直接复制到新数据结果中。

## 9. 数据可视化

本章集中保存各分析模块的核心图表。运行前确保已完成第 5—8 章。

In [ ]:
# Matplotlib 全局样式
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "Arial Unicode MS"]
plt.rcParams["axes.unicode_minus"] = False

# 1. 月度销售趋势
plt.figure(figsize=(14, 6))
plt.plot(monthly_sales.index.astype(str), monthly_sales["销售额"], marker="o")
plt.title("Monthly Sales Trend")
plt.xlabel("Month")
plt.ylabel("Sales (BRL)")
plt.xticks(rotation=45)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(image_path + "monthly_sales_trend.png", dpi=300, bbox_inches="tight")
plt.show()

# 2. 品类销售占比
top8_sales = category_sales["销售额"].head(8)
other_sales = category_sales["销售额"].iloc[8:].sum()
pie_data = pd.concat([top8_sales, pd.Series({"其他品类": other_sales})])
plt.figure(figsize=(9, 9))
plt.pie(pie_data, labels=pie_data.index, autopct="%1.1f%%", startangle=90)
plt.title("Product Category Sales Share")
plt.tight_layout()
plt.savefig(image_path + "category_sales_share.png", dpi=300, bbox_inches="tight")
plt.show()

# 3. 地区销售额 Top 10
plt.figure(figsize=(12, 7))
plt.barh(top10_region.index[::-1], top10_region["销售额"].values[::-1])
plt.title("Top 10 States by Sales")
plt.xlabel("Sales (BRL)")
plt.ylabel("Customer State")
plt.tight_layout()
plt.savefig(image_path + "region_sales_top10.png", dpi=300, bbox_inches="tight")
plt.show()

# 4. 客户购买频次分布
plt.figure(figsize=(10, 6))
plt.bar(purchase_frequency.index.astype(str), purchase_frequency.values)
plt.title("Customer Purchase Frequency")
plt.xlabel("Number of Orders")
plt.ylabel("Number of Customers")
plt.tight_layout()
plt.savefig(image_path + "customer_purchase_frequency.png", dpi=300, bbox_inches="tight")
plt.show()

# 5. 客户价值分层
plt.figure(figsize=(8, 6))
customer_segment["客户数量"].plot(kind="bar")
plt.title("Customer Value Segmentation")
plt.xlabel("Customer Type")
plt.ylabel("Number of Customers")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(image_path + "customer_value_segmentation.png", dpi=300, bbox_inches="tight")
plt.show()

print("核心图表已保存至：", os.path.abspath(image_path))

## 10. 项目总结与业务建议

### 10.1 核心分析发现

请根据本 Notebook 实际运行结果，分别填写：
- **销售表现：** 总商品销售额、已送达订单量、平均客单价及月度趋势。
- **商品品类：** 销售额最高的品类、Top 10 品类销售额占比。
- **地区市场：** 销售额最高的州、Top 10 州销售额占比。
- **客户行为：** 客户复购率、一次性购买客户占比、高价值客户消费贡献。

### 10.2 业务建议

1. **客户留存：** 针对一次性购买客户设计二次购买触达、优惠券或会员权益，并持续监测复购率变化。
2. **高价值客户运营：** 对高消费客户开展分层服务；后续可结合利润、退货和服务成本完善客户价值评估。
3. **商品运营：** 保障核心品类供应，同时结合长尾品类订单量、客单价和利润数据评估资源投入。
4. **区域运营：** 对核心州维持履约体验，对其他州结合订单量、客单价和物流成本制定差异化策略。
5. **进一步分析：** 可扩展支付方式、评价、物流时效和卖家表现分析，补充对销售表现的解释。

### 10.3 项目总结

本项目基于 Olist 巴西电商平台公开数据，使用 Python、Pandas 和 Matplotlib 完成数据质量检查、多表关联、销售指标分析、商品品类分析、地区销售分析、客户行为分析及可视化展示。分析结果可为商品运营、区域市场策略和客户留存提供数据参考。

### 10.4 分析口径与局限性

- 数据范围为 Olist 公开数据集覆盖期间，客户“首次购买”仅指本数据集范围内首次已送达订单。
- 销售额采用商品价格 `price`，不含运费；未计算成本、利润或净收入。
- 客户复购率基于已送达订单计算，不能直接等同于平台全生命周期复购率。
- 高价值客户按累计消费金额分位数识别，尚未纳入利润、退款、服务成本等因素。